In [1]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer


In [2]:
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)


In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
baseline_model = GPT2LMHeadModel.from_pretrained("gpt2")
baseline_model.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
block = baseline_model.transformer.h[0]

print("c_fc bias shape:", block.mlp.c_fc.bias.shape)
print("c_proj bias shape:", block.mlp.c_proj.bias.shape)


c_fc bias shape: torch.Size([3072])
c_proj bias shape: torch.Size([768])


In [5]:
from einops import rearrange

def kronecker_decompose(W, m, n, k=1):
    out_dim, in_dim = W.shape
    m2 = out_dim // m
    n2 = in_dim // n

    W_re = rearrange(
        W,
        '(m m2) (n n2) -> (m n) (m2 n2)',
        m=m, m2=m2, n=n, n2=n2
    )

    U, S, V = torch.svd_lowrank(W_re, q=k)
    A = rearrange(U, '(m n) k -> k m n', m=m, n=n)
    B = rearrange(V, '(m2 n2) k -> k m2 n2', m2=m2, n2=n2)

    scale = S.sqrt().view(-1, 1, 1)
    return A * scale, B * scale


In [6]:
def adaptive_normalize(W, A, B):
    W_hat = torch.kron(A[0], B[0])
    alpha = torch.norm(W, p="fro") / torch.norm(W_hat, p="fro")
    return A * torch.sqrt(alpha), B * torch.sqrt(alpha)t

In [7]:
class KroneckerLinear(nn.Module):
    def __init__(self, A, B, bias):
        super().__init__()
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(B)
        self.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        y = torch.matmul(x, self.A)           
        y = y.unsqueeze(-1) * self.B           
        y = y.reshape(y.shape[0], y.shape[1], -1)
        return y + self.bias


In [8]:
class KroneckerLinearProj(nn.Module):
    def __init__(self, A, B, bias):
        super().__init__()
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(B)
        self.bias = nn.Parameter(bias.clone())

    def forward(self, x):
        x = x.view(x.shape[0], x.shape[1], -1, 2)
        x = (x * self.B).sum(dim=-1)            
        x = torch.matmul(x, self.A.t())         
        return x + self.bias


In [9]:
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

block = model.transformer.h[0]


In [10]:
W_fc = block.mlp.c_fc.weight
b_fc = block.mlp.c_fc.bias

A_fc, B_fc = kronecker_decompose(W_fc, m=768, n=1536)
A_fc, B_fc = adaptive_normalize(W_fc, A_fc, B_fc)

block.mlp.c_fc = KroneckerLinear(A_fc[0], B_fc[0], b_fc)


In [11]:
W_proj = block.mlp.c_proj.weight.T
b_proj = block.mlp.c_proj.bias

A_p, B_p = kronecker_decompose(W_proj, m=768, n=1536)
A_p, B_p = adaptive_normalize(W_proj, A_p, B_p)

block.mlp.c_proj = KroneckerLinearProj(A_p[0], B_p[0], b_proj)


In [12]:
text = "Artificial intelligence is"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    base_logits = baseline_model(**inputs).logits
    comp_logits = model(**inputs).logits

rel_diff = torch.norm(base_logits - comp_logits) / torch.norm(base_logits)
print("Relative logit difference:", rel_diff.item())


Relative logit difference: 0.12810328602790833
